# AR 02 Boletin / Gazette Timeline Builder

**Author:** Noah Green CPA CFE

This notebook demonstrates how to turn synthetic Argentina official-gazette-style notice rows into a dated source-literacy timeline. It uses only synthetic/redacted inputs and does not make a legal conclusion, assign jurisdiction-level scoring, or treat jurisdiction as a proxy for risk.

## Source-Literacy Objective

The task is to show the diligence mechanics: load a small notice-style dataset, use the shared timeline builder, and export a compact table that helps an analyst see the order of public-source leads. The table is a reading aid only. If the legal effect of a notice matters for a transaction, contract condition, ownership issue, regulatory representation, or closing decision, escalate to Argentina local counsel before relying on it.

In [1]:
from __future__ import annotations

import csv
import sys
from pathlib import Path


def find_lab_root() -> Path:
    relative_lab_root = Path(
        "SPP/articles/2026-06-14_cross_border_open_source_diligence_atlas/"
        "lab/cross-border-diligence-atlas"
    )
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / relative_lab_root]
    for candidate in candidates:
        if (candidate / "src" / "crossborder_dd").is_dir() and (candidate / "pyproject.toml").exists():
            return candidate.resolve()
    raise RuntimeError("Run from the lab root, notebook directory, or NGO repository root.")


LAB_ROOT = find_lab_root()
SRC_PATH = LAB_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from crossborder_dd.export_csv import write_rows
from crossborder_dd.gazette_timeline_builder import build_timeline

SYNTHETIC_EVENTS_PATH = LAB_ROOT / "data" / "synthetic" / "gazette_events.csv"
OUTPUT_PATH = LAB_ROOT / "data" / "redacted_outputs" / "AR_02_boletin_timeline_builder.csv"

LAB_ROOT

PosixPath('/Users/noah.r.green/dd-tech-lab-companions/cross_border_diligence_atlas')

In [2]:
with SYNTHETIC_EVENTS_PATH.open(newline="", encoding="utf-8") as handle:
    synthetic_notice_rows = list(csv.DictReader(handle))

synthetic_notice_rows

[{'event_date': '2026-02-10',
  'entity_name': 'Sociedad Sintetica Argentina SA',
  'event_type': 'synthetic_notice',
  'description': 'Synthetic official-gazette notice for timeline testing',
  'source_url': 'https://example.invalid/synthetic-notice-1'},
 {'event_date': '2026-01-15',
  'entity_name': 'Sociedad Sintetica Argentina SA',
  'event_type': 'synthetic_notice',
  'description': 'Earlier synthetic official-gazette notice for timeline testing',
  'source_url': 'https://example.invalid/synthetic-notice-2'}]

In [3]:
timeline = build_timeline(synthetic_notice_rows, "Sociedad Sintetica Argentina SA")

assert [event["event_date"] for event in timeline] == sorted(event["event_date"] for event in timeline)
assert all(event["limitation"] for event in timeline)

write_rows(OUTPUT_PATH, timeline)
timeline

[{'event_date': '2026-01-15',
  'entity_name': 'Sociedad Sintetica Argentina SA',
  'event_type': 'synthetic_notice',
  'description': 'Earlier synthetic official-gazette notice for timeline testing',
  'source_url': 'https://example.invalid/synthetic-notice-2',
  'limitation': 'Timeline lead only. Verify notice text in the official source.'},
 {'event_date': '2026-02-10',
  'entity_name': 'Sociedad Sintetica Argentina SA',
  'event_type': 'synthetic_notice',
  'description': 'Synthetic official-gazette notice for timeline testing',
  'source_url': 'https://example.invalid/synthetic-notice-1',
  'limitation': 'Timeline lead only. Verify notice text in the official source.'}]

In [4]:
def markdown_table(rows: list[dict[str, object]], columns: list[str]) -> str:
    header = "| " + " | ".join(columns) + " |"
    divider = "| " + " | ".join("---" for _ in columns) + " |"
    body = [
        "| " + " | ".join(str(row.get(column, "")).replace("|", "\\|") for column in columns) + " |"
        for row in rows
    ]
    return "\n".join([header, divider, *body])


display_columns = ["event_date", "entity_name", "event_type", "description", "limitation"]
print(markdown_table(timeline, display_columns))

| event_date | entity_name | event_type | description | limitation |
| --- | --- | --- | --- | --- |
| 2026-01-15 | Sociedad Sintetica Argentina SA | synthetic_notice | Earlier synthetic official-gazette notice for timeline testing | Timeline lead only. Verify notice text in the official source. |
| 2026-02-10 | Sociedad Sintetica Argentina SA | synthetic_notice | Synthetic official-gazette notice for timeline testing | Timeline lead only. Verify notice text in the official source. |


## What This Proves / What It Cannot Prove

**What this proves:** the synthetic Boletin/gazette-style rows can be loaded from `data/synthetic/gazette_events.csv`, passed through the shared `build_timeline` function, sorted into chronology, and exported as a redacted source-literacy table at `data/redacted_outputs/AR_02_boletin_timeline_builder.csv`.

**What it cannot prove:** the table does not verify that any real notice exists, establish legal status, interpret the legal effect of a publication, identify adverse conduct, or support jurisdiction-level scoring. For real work, each notice must be read in the official source, matched to the correct entity, translated with care where needed, and escalated to Argentina local counsel whenever legal effect or transaction reliance matters.